# PyTorch Tutorial 41: Advanced Preference Optimization (The FAANG Standard)

Tutorial 19 introduced DPO with a loss function. Tutorial 40 showed the full PPO pipeline. Now we go **deep** into the modern preference optimization landscape.

This tutorial covers every major algorithm you need to know: **DPO** (full training loop), **IPO**, **KTO**, **ORPO**, and **GRPO** (DeepSeek-R1's breakthrough algorithm). All implemented from scratch.

## Learning Objectives
1. **Compute log-probabilities** from a language model — the foundation of all preference methods
2. **Build a full DPO training loop** — not just the loss, the entire pipeline
3. **Implement 4 DPO variants** from scratch — IPO, KTO, ORPO
4. **Master GRPO** — DeepSeek-R1's algorithm that eliminates the critic network
5. **Compare all methods** — when to use which, tradeoffs, and interview answers

**Prerequisites**: Tutorial 19 (DPO loss), Tutorial 40 (Reward Model, PPO)

---

## 1. Vocabulary First

- **DPO (Direct Preference Optimization)**: Optimizes policy directly on preference pairs. No reward model needed. Loss: -log sigmoid(beta * (log_ratio_chosen - log_ratio_rejected)).
- **IPO (Identity Preference Optimization)**: Replaces log-sigmoid with squared loss. More robust to noisy preferences.
- **KTO (Kahneman-Tversky Optimization)**: Doesn't need paired data — just (prompt, response, good/bad). Based on prospect theory.
- **ORPO (Odds Ratio Preference Optimization)**: Uses odds ratio instead of log ratio. Combines SFT and alignment in one step.
- **GRPO (Group Relative Policy Optimization)**: DeepSeek-R1's algorithm. No critic/value network. Samples K responses per prompt, uses group-relative ranking as advantage.
- **Online DPO**: Generate fresh responses each step (vs offline = fixed dataset).
- **Beta (temperature)**: Controls how much the policy can deviate from the reference. Higher = more conservative.
- **Implicit Reward**: In DPO, the reward is implicit: r(x,y) = beta * log(pi(y|x) / pi_ref(y|x)).

### Method Comparison Table

| Method | Data Format | Reward Model | Critic | Key Innovation |
|--------|------------|-------------|--------|----------------|
| PPO | prompts only | Yes | Yes | Online generation + RL |
| DPO | (prompt, chosen, rejected) | No | No | Closed-form RL solution |
| IPO | (prompt, chosen, rejected) | No | No | Squared loss, robust |
| KTO | (prompt, response, good/bad) | No | No | No paired data needed |
| ORPO | (prompt, chosen, rejected) | No | No | Combines SFT + alignment |
| GRPO | prompts only | Yes (or verifier) | No | Group-relative advantages |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import copy
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Ready for Advanced Preference Optimization!")

---

## 2. Part 1: Computing Log-Probabilities from a Language Model

Every preference optimization method needs **log-probabilities** of sequences. This is the foundation.

Given a sequence [t1, t2, t3, ...], the log-probability is:

```
log P(sequence) = log P(t1) + log P(t2|t1) + log P(t3|t1,t2) + ...
                = sum of per-token log probs
```

**Key detail**: We use the **label-shifted** logits. At position i, the logits predict token i+1.

### FAANG Interview Question

**Q: "How do you compute sequence log-probability from a language model?"**

**A**: Forward pass the entire sequence to get logits at each position. Apply log_softmax to get per-token log probabilities. Then gather the log-prob of the *actual* next token at each position (shift by 1). Sum across the response tokens (not the prompt). Mask out padding tokens. The result is the total log-probability of generating that specific response given the prompt.

In [ ]:
class SmallLM(nn.Module):
    """Small causal language model for preference optimization demos.
    
    Same architecture as Tutorial 40 but without the value head
    (preference methods don't need a critic).
    """
    
    def __init__(self, vocab_size=1000, embed_dim=128, n_heads=4,
                 n_layers=2, max_seq_len=64):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )
        self.lm_head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, input_ids):
        """Returns logits for next-token prediction.
        
        Args:
            input_ids: [batch, seq_len]
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_ids.device), diagonal=1
        ).bool()
        x = self.transformer(x, mask=causal_mask)
        return self.lm_head(x)


def compute_log_probs(model, input_ids, response_mask):
    """Compute per-token and total log-probabilities of a sequence.
    
    This is THE core function for all preference optimization methods.
    
    Args:
        model: language model returning logits [batch, seq_len, vocab]
        input_ids: [batch, seq_len] full sequence (prompt + response)
        response_mask: [batch, seq_len] 1 for response tokens, 0 for prompt/padding
    Returns:
        total_log_probs: [batch] sum of log probs over response tokens
        per_token_log_probs: [batch, seq_len] log prob at each position
    """
    # Step 1: Forward pass to get logits
    logits = model(input_ids)  # [batch, seq_len, vocab]
    
    # Step 2: Shift logits and labels (standard causal LM pattern)
    # logits[i] predicts token[i+1]
    shift_logits = logits[:, :-1, :]   # [batch, seq_len-1, vocab]
    shift_labels = input_ids[:, 1:]     # [batch, seq_len-1]
    shift_mask = response_mask[:, 1:]   # [batch, seq_len-1]
    
    # Step 3: Compute per-token log probabilities
    log_probs = F.log_softmax(shift_logits, dim=-1)  # [batch, seq_len-1, vocab]
    
    # Step 4: Gather the log prob of the actual next token
    per_token = log_probs.gather(
        2, shift_labels.unsqueeze(-1)
    ).squeeze(-1)  # [batch, seq_len-1]
    
    # Step 5: Mask and sum (only response tokens count)
    masked = per_token * shift_mask
    total = masked.sum(dim=-1)  # [batch]
    
    # Pad per_token back to original seq_len for convenience
    per_token_full = F.pad(per_token, (1, 0), value=0.0)  # [batch, seq_len]
    
    return total, per_token_full


# Demonstrate
model = SmallLM(vocab_size=1000, embed_dim=128).to(device)
test_ids = torch.randint(0, 1000, (2, 20)).to(device)
# Assume first 8 tokens are prompt, rest are response
test_mask = torch.zeros(2, 20).to(device)
test_mask[:, 8:] = 1.0

total_lp, per_token_lp = compute_log_probs(model, test_ids, test_mask)
print(f"Total log-probs: {total_lp.detach().cpu().numpy()}")
print(f"Per-token shape: {per_token_lp.shape}")
print(f"\nSample per-token log-probs (response only):")
print(f"  {per_token_lp[0, 8:].detach().cpu().numpy().round(3)}")

---

## 3. Part 2: Full DPO Training Loop

Tutorial 19 gave us the loss function. Now we build the complete training pipeline.

### FAANG Interview Question

**Q: "Walk through one DPO training step. What are the 4 forward passes?"**

**A**:
1. Forward chosen through **policy** -> log_probs_chosen_policy
2. Forward rejected through **policy** -> log_probs_rejected_policy
3. Forward chosen through **reference** (no grad) -> log_probs_chosen_ref
4. Forward rejected through **reference** (no grad) -> log_probs_rejected_ref

Then: loss = -log sigmoid(beta * ((policy_chosen - ref_chosen) - (policy_rejected - ref_rejected)))

The reference model is frozen. Only the policy model gets gradient updates.

In [ ]:
class PreferenceDataset(Dataset):
    """Preference dataset with (prompt, chosen, rejected) triples.
    
    Uses synthetic data where 'chosen' has a learnable pattern.
    In production, this would be human-annotated preference pairs.
    """
    
    def __init__(self, n_samples=500, prompt_len=8, resp_len=12, vocab_size=1000):
        self.data = []
        self.prompt_len = prompt_len
        self.resp_len = resp_len
        
        for _ in range(n_samples):
            prompt = torch.randint(0, vocab_size, (prompt_len,))
            chosen = torch.randint(0, vocab_size // 2, (resp_len,))
            rejected = torch.randint(vocab_size // 2, vocab_size, (resp_len,))
            self.data.append((prompt, chosen, rejected))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        prompt, chosen, rejected = self.data[idx]
        # Concatenate prompt + response
        chosen_seq = torch.cat([prompt, chosen])
        rejected_seq = torch.cat([prompt, rejected])
        # Create response masks
        mask = torch.zeros(self.prompt_len + self.resp_len)
        mask[self.prompt_len:] = 1.0
        return chosen_seq, rejected_seq, mask


class DPOTrainer:
    """Full DPO training pipeline.
    
    Manages policy model, reference model, optimizer,
    and tracks training metrics.
    """
    
    def __init__(self, policy, ref_model, beta=0.1, lr=1e-4):
        self.policy = policy
        self.ref = ref_model
        self.beta = beta
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        
        # Freeze reference
        for p in self.ref.parameters():
            p.requires_grad = False
        
        self.history = {"loss": [], "accuracy": [], 
                        "reward_margin": [], "chosen_reward": [], "rejected_reward": []}
    
    def dpo_loss(self, policy_chosen_lp, policy_rejected_lp,
                 ref_chosen_lp, ref_rejected_lp):
        """DPO loss from Tutorial 19, now with full context."""
        logr_chosen = policy_chosen_lp - ref_chosen_lp
        logr_rejected = policy_rejected_lp - ref_rejected_lp
        logits = self.beta * (logr_chosen - logr_rejected)
        loss = -F.logsigmoid(logits).mean()
        
        with torch.no_grad():
            acc = (logr_chosen > logr_rejected).float().mean()
            chosen_reward = self.beta * logr_chosen
            rejected_reward = self.beta * logr_rejected
        
        return loss, acc, chosen_reward.mean(), rejected_reward.mean()
    
    def train_epoch(self, dataloader):
        """Train for one epoch. Returns average metrics."""
        self.policy.train()
        epoch_metrics = {k: [] for k in self.history}
        
        for chosen_seq, rejected_seq, mask in dataloader:
            chosen_seq = chosen_seq.to(device)
            rejected_seq = rejected_seq.to(device)
            mask = mask.to(device)
            
            # 4 forward passes
            policy_chosen_lp, _ = compute_log_probs(self.policy, chosen_seq, mask)
            policy_rejected_lp, _ = compute_log_probs(self.policy, rejected_seq, mask)
            
            with torch.no_grad():
                ref_chosen_lp, _ = compute_log_probs(self.ref, chosen_seq, mask)
                ref_rejected_lp, _ = compute_log_probs(self.ref, rejected_seq, mask)
            
            loss, acc, c_rew, r_rew = self.dpo_loss(
                policy_chosen_lp, policy_rejected_lp,
                ref_chosen_lp, ref_rejected_lp
            )
            
            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
            self.optimizer.step()
            
            epoch_metrics["loss"].append(loss.item())
            epoch_metrics["accuracy"].append(acc.item())
            epoch_metrics["reward_margin"].append((c_rew - r_rew).item())
            epoch_metrics["chosen_reward"].append(c_rew.item())
            epoch_metrics["rejected_reward"].append(r_rew.item())
        
        for k in self.history:
            avg = np.mean(epoch_metrics[k])
            self.history[k].append(avg)
        
        return {k: self.history[k][-1] for k in self.history}


# Train DPO
print("Training DPO...")
print("=" * 50)

VOCAB_SIZE = 1000
policy = SmallLM(VOCAB_SIZE, embed_dim=128).to(device)
ref = copy.deepcopy(policy).to(device)

dpo_trainer = DPOTrainer(policy, ref, beta=0.1, lr=1e-4)
dataset = PreferenceDataset(n_samples=400, vocab_size=VOCAB_SIZE)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for epoch in range(8):
    metrics = dpo_trainer.train_epoch(loader)
    if epoch % 2 == 0:
        print(f"Epoch {epoch}: Loss={metrics['loss']:.4f}, "
              f"Acc={metrics['accuracy']:.2%}, "
              f"Margin={metrics['reward_margin']:.4f}")

print("\nDPO training complete!")

In [ ]:
# Visualize DPO training
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(dpo_trainer.history['loss'], 'b-', linewidth=2)
axes[0].set_title('DPO Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(dpo_trainer.history['accuracy'], 'g-', linewidth=2)
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
axes[1].set_title('Preference Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(dpo_trainer.history['chosen_reward'], 'g-', linewidth=2, label='Chosen')
axes[2].plot(dpo_trainer.history['rejected_reward'], 'r-', linewidth=2, label='Rejected')
axes[2].fill_between(
    range(len(dpo_trainer.history['chosen_reward'])),
    dpo_trainer.history['chosen_reward'],
    dpo_trainer.history['rejected_reward'],
    alpha=0.15, color='blue'
)
axes[2].set_title('Implicit Reward Margin', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Implicit Reward')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 4. Part 3: DPO Variants (IPO, KTO, ORPO)

DPO has several variants, each addressing a specific limitation:

- **IPO**: Replaces log-sigmoid with squared loss. More robust to noisy/conflicting preferences.
- **KTO**: Doesn't need **paired** data. Just (prompt, response, thumbs_up/thumbs_down). Cheaper to collect.
- **ORPO**: Combines SFT and alignment into a single training step using odds ratios.

In [ ]:
def dpo_loss_fn(policy_chosen_lp, policy_rejected_lp,
                ref_chosen_lp, ref_rejected_lp, beta=0.1):
    """Standard DPO loss. Loss = -log sigmoid(beta * margin)."""
    logr_chosen = policy_chosen_lp - ref_chosen_lp
    logr_rejected = policy_rejected_lp - ref_rejected_lp
    return -F.logsigmoid(beta * (logr_chosen - logr_rejected)).mean()


def ipo_loss_fn(policy_chosen_lp, policy_rejected_lp,
                ref_chosen_lp, ref_rejected_lp, beta=0.1, tau=0.5):
    """IPO (Identity Preference Optimization) loss.
    
    Replaces log-sigmoid with squared loss for robustness to noise.
    Loss = (log_ratio_margin - 1/(2*beta))^2
    
    Key insight: When preferences are noisy (annotators disagree),
    the sigmoid in DPO can over-fit to the noise. IPO's squared loss
    is more forgiving.
    """
    logr_chosen = policy_chosen_lp - ref_chosen_lp
    logr_rejected = policy_rejected_lp - ref_rejected_lp
    margin = logr_chosen - logr_rejected
    target = 1.0 / (2.0 * beta)
    return ((margin - target) ** 2).mean()


def kto_loss_fn(policy_lp, ref_lp, is_good, beta=0.1):
    """KTO (Kahneman-Tversky Optimization) loss.
    
    No paired data needed! Just (prompt, response, good/bad).
    Based on prospect theory: losses loom larger than gains.
    
    For good responses: loss = 1 - sigmoid(beta * (log_ratio - KL_ref))
    For bad responses:  loss = 1 - sigmoid(beta * (KL_ref - log_ratio))
    
    Args:
        policy_lp: [batch] policy log-probs
        ref_lp: [batch] reference log-probs
        is_good: [batch] 1.0 for good responses, 0.0 for bad
        beta: temperature
    """
    log_ratio = policy_lp - ref_lp
    
    # KL estimate (mean log ratio as a proxy)
    kl_ref = log_ratio.detach().mean()
    
    # Good responses: increase probability (relative to KL)
    good_loss = 1.0 - torch.sigmoid(beta * (log_ratio - kl_ref))
    
    # Bad responses: decrease probability (relative to KL)
    bad_loss = 1.0 - torch.sigmoid(beta * (kl_ref - log_ratio))
    
    # Weight: prospect theory says losses (bad) weigh more
    loss_weight_good = 1.0
    loss_weight_bad = 1.5  # loss aversion coefficient
    
    loss = (is_good * loss_weight_good * good_loss + 
            (1.0 - is_good) * loss_weight_bad * bad_loss)
    
    return loss.mean()


def orpo_loss_fn(policy_chosen_lp, policy_rejected_lp, beta=0.1):
    """ORPO (Odds Ratio Preference Optimization) loss.
    
    No reference model needed! Combines SFT + alignment.
    Uses odds ratio: odds(y) = P(y) / (1 - P(y))
    
    Loss = SFT_loss + beta * -log sigmoid(log(odds_chosen / odds_rejected))
    
    Here we only compute the preference part (SFT is separate).
    """
    # Convert log-probs to average per-token log-prob, then to probability
    # (Using a rough approximation for demo purposes)
    chosen_avg_lp = policy_chosen_lp / 12.0  # divide by response length
    rejected_avg_lp = policy_rejected_lp / 12.0
    
    # Log odds ratio
    # log(odds_chosen / odds_rejected) = log_odds_chosen - log_odds_rejected
    # where log_odds = log_p - log(1-p) approx= log_p - log(-log_p) for small p
    log_odds_ratio = chosen_avg_lp - rejected_avg_lp
    
    return -F.logsigmoid(beta * log_odds_ratio).mean()


# Test all loss functions
print("Testing all preference loss functions:")
print("=" * 50)

# Dummy data: policy prefers chosen correctly
p_chosen = torch.tensor([-5.0, -6.0, -4.0])
p_rejected = torch.tensor([-8.0, -9.0, -7.0])
r_chosen = torch.tensor([-5.5, -6.5, -4.5])
r_rejected = torch.tensor([-5.5, -6.5, -4.5])

print(f"DPO loss:  {dpo_loss_fn(p_chosen, p_rejected, r_chosen, r_rejected):.4f}")
print(f"IPO loss:  {ipo_loss_fn(p_chosen, p_rejected, r_chosen, r_rejected):.4f}")
print(f"KTO loss:  {kto_loss_fn(p_chosen, r_chosen, torch.ones(3)):.4f} (good)")
print(f"KTO loss:  {kto_loss_fn(p_rejected, r_rejected, torch.zeros(3)):.4f} (bad)")
print(f"ORPO loss: {orpo_loss_fn(p_chosen, p_rejected):.4f}")

print("\nAll losses computed successfully!")

In [ ]:
# Visualize loss landscapes for all methods
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create a range of margins (chosen_lp - rejected_lp relative to reference)
margins = torch.linspace(-3, 3, 200)

beta = 0.5

# DPO: -log sigmoid(beta * margin)
dpo_vals = -F.logsigmoid(beta * margins).numpy()
axes[0, 0].plot(margins.numpy(), dpo_vals, 'b-', linewidth=2)
axes[0, 0].set_title('DPO Loss', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Log-ratio margin (chosen - rejected)')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].annotate('Model prefers rejected', xy=(-2.5, 3), fontsize=9, color='red')
axes[0, 0].annotate('Model prefers chosen', xy=(1.0, 0.5), fontsize=9, color='green')

# IPO: (margin - 1/(2*beta))^2
target = 1.0 / (2.0 * beta)
ipo_vals = ((margins - target) ** 2).numpy()
axes[0, 1].plot(margins.numpy(), ipo_vals, 'r-', linewidth=2)
axes[0, 1].axvline(x=target, color='green', linestyle='--', alpha=0.7, label=f'Target={target:.1f}')
axes[0, 1].set_title('IPO Loss (Squared)', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Log-ratio margin')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# KTO: 1 - sigmoid(beta * (ratio - KL))
kl_proxy = 0.0
kto_good = (1.0 - torch.sigmoid(beta * (margins - kl_proxy))).numpy()
kto_bad = (1.0 - torch.sigmoid(beta * (kl_proxy - margins))).numpy()
axes[1, 0].plot(margins.numpy(), kto_good, 'g-', linewidth=2, label='Good responses')
axes[1, 0].plot(margins.numpy(), kto_bad, 'r-', linewidth=2, label='Bad responses')
axes[1, 0].set_title('KTO Loss', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Log-ratio (policy vs reference)')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# ORPO: -log sigmoid(beta * margin) (similar shape to DPO but without ref)
orpo_vals = -F.logsigmoid(beta * margins / 12.0).numpy()  # per-token avg
axes[1, 1].plot(margins.numpy(), orpo_vals, 'purple', linewidth=2)
axes[1, 1].set_title('ORPO Loss', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Log-prob margin (chosen - rejected)')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Loss Landscapes: DPO vs IPO vs KTO vs ORPO', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key differences:")
print("  DPO:  Sigmoid - smooth but can overfit to noise")
print("  IPO:  Squared - has a finite minimum, more robust")
print("  KTO:  Asymmetric - different curves for good vs bad")
print("  ORPO: No reference model - simpler but less controlled")

---

## 5. Part 4: GRPO (Group Relative Policy Optimization)

**GRPO is the algorithm behind DeepSeek-R1** — one of the most significant advances in post-training RL.

### Key Insight

PPO needs a **critic** (value network) to compute advantages. GRPO eliminates the critic entirely:

1. For each prompt, sample **K responses** from the policy
2. Score each response with a reward model (or verifier)
3. Compute **group-relative advantages**: normalize rewards within the group
4. Use PPO-style clipped objective with these advantages

```
Advantage_i = (reward_i - mean(rewards)) / std(rewards)
```

This is simple, effective, and avoids the value function estimation problem entirely.

### FAANG Interview Question

**Q: "How does GRPO differ from PPO? Why does it work?"**

**A**: GRPO replaces the learned value function (critic) with a simple statistical normalization over a group of sampled responses. Instead of V(s) predicting expected reward, GRPO computes the mean/std of actual rewards from K samples per prompt. This is an unbiased estimate of the advantage — if a response scores above the group mean, it gets positive advantage. This works because: (1) you're comparing within-prompt, so the baseline is always relevant, (2) the critic in PPO is often inaccurate early in training anyway, (3) it eliminates value function estimation errors that can destabilize training.

In [ ]:
def grpo_loss(new_log_probs, old_log_probs, advantages,
              ref_log_probs=None, epsilon=0.2, beta=0.01):
    """GRPO loss function (DeepSeek-R1 style).
    
    Combines PPO's clipped objective with group-relative advantages
    and an optional KL penalty against the reference.
    
    Args:
        new_log_probs: [batch] current policy log-probs
        old_log_probs: [batch] old policy log-probs (from sampling)
        advantages: [batch] group-normalized advantages
        ref_log_probs: [batch] optional reference log-probs for KL penalty
        epsilon: PPO clipping parameter
        beta: KL penalty coefficient
    Returns:
        loss: scalar
        metrics: dict with components
    """
    # PPO clipped objective
    ratio = torch.exp(new_log_probs - old_log_probs)
    obj_unclipped = ratio * advantages
    obj_clipped = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    policy_loss = -torch.min(obj_unclipped, obj_clipped).mean()
    
    # KL penalty (optional but recommended)
    kl_loss = 0.0
    if ref_log_probs is not None:
        kl = (new_log_probs - ref_log_probs).mean()
        kl_loss = beta * kl
    
    total_loss = policy_loss + kl_loss
    
    return total_loss, {
        "policy_loss": policy_loss.item(),
        "kl_loss": kl_loss if isinstance(kl_loss, float) else kl_loss.item(),
        "mean_ratio": ratio.mean().item()
    }


def compute_group_advantages(rewards, group_size):
    """Compute group-relative advantages (GRPO's core innovation).
    
    For each group of K responses to the same prompt,
    normalize rewards to zero mean, unit variance.
    
    Args:
        rewards: [n_prompts * group_size] flat tensor of rewards
        group_size: K responses per prompt
    Returns:
        advantages: [n_prompts * group_size] normalized advantages
    """
    # Reshape into groups
    n_prompts = len(rewards) // group_size
    grouped = rewards[:n_prompts * group_size].reshape(n_prompts, group_size)
    
    # Normalize within each group
    group_mean = grouped.mean(dim=1, keepdim=True)
    group_std = grouped.std(dim=1, keepdim=True) + 1e-8
    normalized = (grouped - group_mean) / group_std
    
    return normalized.reshape(-1)


# Demonstrate group advantages
print("GRPO Group Advantages Demo:")
print("=" * 50)

# 3 prompts, 4 responses each
demo_rewards = torch.tensor([
    0.8, 0.3, 0.1, 0.5,  # Prompt 1: response 0 is best
    0.2, 0.9, 0.4, 0.6,  # Prompt 2: response 1 is best
    0.5, 0.5, 0.5, 0.5,  # Prompt 3: all equal (advantages = 0)
])

advantages = compute_group_advantages(demo_rewards, group_size=4)
for i in range(3):
    r = demo_rewards[i*4:(i+1)*4].numpy().round(2)
    a = advantages[i*4:(i+1)*4].numpy().round(3)
    print(f"  Prompt {i+1}: Rewards={r} -> Advantages={a}")

print("\nNotice: Prompt 3 has zero advantages (all responses equal).")

In [ ]:
class GRPOTrainer:
    """GRPO Trainer implementing the DeepSeek-R1 approach.
    
    Key difference from PPO: no value network.
    Instead, sample K responses per prompt and use group-relative advantages.
    """
    
    def __init__(self, policy, ref_model, reward_fn,
                 group_size=4, lr=1e-4, epsilon=0.2, beta=0.01):
        self.policy = policy
        self.ref = ref_model
        self.reward_fn = reward_fn  # function that scores responses
        self.group_size = group_size
        self.epsilon = epsilon
        self.beta = beta
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        
        for p in self.ref.parameters():
            p.requires_grad = False
        
        self.history = []
    
    def train_step(self, prompts, max_new_tokens=12):
        """One GRPO training step.
        
        1. Generate K responses per prompt
        2. Score with reward function
        3. Compute group-relative advantages
        4. PPO-style update
        """
        n_prompts = prompts.shape[0]
        prompt_len = prompts.shape[1]
        
        # Step 1: Repeat each prompt K times
        expanded = prompts.repeat_interleave(self.group_size, dim=0)
        
        # Step 2: Generate responses (collect old log-probs)
        self.policy.train(False)
        with torch.no_grad():
            all_seqs = expanded.clone()
            old_log_probs_list = []
            
            for _ in range(max_new_tokens):
                logits = self.policy(all_seqs)
                next_logits = logits[:, -1, :]
                probs = F.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, 1)
                lp = F.log_softmax(next_logits, dim=-1)
                old_lp = lp.gather(1, next_token).squeeze(-1)
                old_log_probs_list.append(old_lp)
                all_seqs = torch.cat([all_seqs, next_token], dim=1)
            
            old_log_probs = torch.stack(old_log_probs_list, dim=1).sum(dim=1)
        
        # Step 3: Score responses
        with torch.no_grad():
            rewards = self.reward_fn(all_seqs)
        
        # Step 4: Group-relative advantages
        advantages = compute_group_advantages(rewards, self.group_size)
        
        # Step 5: PPO update
        self.policy.train(True)
        
        # Recompute log probs (with gradients)
        resp_mask = torch.zeros_like(all_seqs, dtype=torch.float)
        resp_mask[:, prompt_len:] = 1.0
        new_log_probs, _ = compute_log_probs(self.policy, all_seqs, resp_mask)
        
        with torch.no_grad():
            ref_log_probs, _ = compute_log_probs(self.ref, all_seqs, resp_mask)
        
        loss, loss_metrics = grpo_loss(
            new_log_probs, old_log_probs, advantages,
            ref_log_probs=ref_log_probs,
            epsilon=self.epsilon, beta=self.beta
        )
        
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        
        step_metrics = {
            "mean_reward": rewards.mean().item(),
            "reward_std": rewards.std().item(),
            "loss": loss.item(),
            **loss_metrics
        }
        self.history.append(step_metrics)
        return step_metrics


# Train GRPO
print("Training GRPO (DeepSeek-R1 style)...")
print("=" * 50)

# Simple reward function: prefer lower token IDs in response
def simple_reward(sequences):
    resp = sequences[:, 8:]  # skip prompt
    return -(resp.float().mean(dim=1) / 1000.0)  # lower tokens = higher reward

grpo_policy = SmallLM(VOCAB_SIZE, embed_dim=128).to(device)
grpo_ref = copy.deepcopy(grpo_policy).to(device)

grpo_trainer = GRPOTrainer(
    policy=grpo_policy, ref_model=grpo_ref,
    reward_fn=simple_reward, group_size=4,
    lr=1e-4, epsilon=0.2, beta=0.01
)

for step in range(8):
    prompts = torch.randint(0, VOCAB_SIZE, (6, 8)).to(device)
    metrics = grpo_trainer.train_step(prompts, max_new_tokens=12)
    if step % 2 == 0:
        print(f"Step {step}: Reward={metrics['mean_reward']:.4f}, "
              f"Loss={metrics['loss']:.4f}")

print("\nGRPO training complete!")

In [ ]:
# GRPO training visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

grpo_steps = range(len(grpo_trainer.history))
grpo_rewards = [h['mean_reward'] for h in grpo_trainer.history]
grpo_losses = [h['loss'] for h in grpo_trainer.history]
grpo_ratios = [h['mean_ratio'] for h in grpo_trainer.history]

axes[0].plot(grpo_steps, grpo_rewards, 'g-o', linewidth=2)
axes[0].set_title('GRPO: Mean Reward', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Reward')
axes[0].grid(True, alpha=0.3)

axes[1].plot(grpo_steps, grpo_losses, 'b-o', linewidth=2)
axes[1].set_title('GRPO: Total Loss', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

axes[2].plot(grpo_steps, grpo_ratios, 'r-o', linewidth=2)
axes[2].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='No change')
axes[2].set_title('GRPO: Mean Policy Ratio', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Ratio (new/old)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('GRPO Training (DeepSeek-R1 Algorithm)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 6. Part 5: Beta Sensitivity Analysis

### FAANG Interview Question

**Q: "How does beta affect DPO training? How would you tune it?"**

**A**: Beta controls the KL constraint strength. Higher beta = stay closer to reference (more conservative). Lower beta = optimize preferences more aggressively (risk of reward hacking). Symptoms of wrong beta:
- **Too high**: Loss barely decreases, accuracy improves slowly, policy barely changes
- **Too low**: Loss decreases fast, but output quality degrades (model finds adversarial solutions)

Typical range: beta in [0.05, 0.5]. Start with 0.1. If accuracy is too low, decrease beta. If output quality degrades, increase beta.

In [ ]:
# Beta sensitivity experiment
print("Beta Sensitivity Experiment")
print("=" * 50)

betas = [0.01, 0.1, 0.5, 1.0]
beta_results = {}

dataset = PreferenceDataset(n_samples=300, vocab_size=VOCAB_SIZE)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for beta_val in betas:
    policy = SmallLM(VOCAB_SIZE, embed_dim=128).to(device)
    ref = copy.deepcopy(policy).to(device)
    trainer = DPOTrainer(policy, ref, beta=beta_val, lr=1e-4)
    
    for epoch in range(8):
        trainer.train_epoch(loader)
    
    beta_results[beta_val] = dict(trainer.history)
    print(f"  beta={beta_val}: Final acc={trainer.history['accuracy'][-1]:.2%}, "
          f"margin={trainer.history['reward_margin'][-1]:.4f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ['red', 'blue', 'green', 'purple']

for beta_val, color in zip(betas, colors):
    axes[0].plot(beta_results[beta_val]['loss'], color=color, 
                 linewidth=2, label=f'beta={beta_val}')
    axes[1].plot(beta_results[beta_val]['accuracy'], color=color, linewidth=2)
    axes[2].plot(beta_results[beta_val]['reward_margin'], color=color, linewidth=2)

axes[0].set_title('DPO Loss vs Beta', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Accuracy vs Beta', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
axes[1].grid(True, alpha=0.3)

axes[2].set_title('Reward Margin vs Beta', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Margin (chosen - rejected)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Beta Sensitivity: How Temperature Affects DPO', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nLower beta = faster learning but potentially less stable.")
print("Higher beta = slower but safer optimization.")

---

## 7. FAANG Interview Questions

### Q1: "DPO vs RLHF (PPO): What are the real tradeoffs?"

**A**:
- **DPO wins**: Simpler (no reward model, no RL loop), fewer hyperparameters, less compute (2 models vs 4), more stable training.
- **PPO wins**: Online data (generates fresh responses), explicit reward signal (reusable RM), often better at scale (OpenAI/Anthropic still use PPO for final stage).
- **Key insight**: DPO is offline (fixed dataset), so it can't explore beyond the training distribution. PPO generates new responses and can discover novel good behaviors.

---

### Q2: "What is the implicit reward in DPO?"

**A**: DPO implicitly defines a reward function: r(x, y) = beta * log(pi(y|x) / pi_ref(y|x)) + constant. The model learns to assign higher probability (relative to reference) to preferred responses. We never compute this reward explicitly, but we can extract it for analysis. This is why DPO doesn't need a reward model — the reward is encoded in the policy's probability distribution relative to the reference.

---

### Q3: "How does GRPO eliminate the critic? Won't this increase variance?"

**A**: GRPO replaces the learned critic with a group-level statistic (mean/std of K sampled rewards). This actually *reduces* variance in early training because:
1. The critic is most inaccurate early on (chicken-and-egg: needs good data to train, but needs good critic to collect data)
2. Group normalization is an unbiased estimator of advantage (by construction)
3. The variance from K samples decreases as K increases (typically K=4-8)

The tradeoff: GRPO requires K forward passes per prompt (more compute per step) but eliminates critic training entirely. Net effect: simpler and often faster to converge.

---

### Q4: "When would you use KTO over DPO?"

**A**: KTO when you have unpaired binary feedback (thumbs up/down from users). DPO requires explicit A-is-better-than-B pairs, which are expensive to collect. KTO only needs "this response is good" or "this response is bad". Real-world scenario: you have user thumbs up/down data but didn't show them side-by-side comparisons.

---

### Q5: "You're debugging DPO and accuracy is stuck at 55%. What do you check?"

**A**:
1. **Data quality**: Are the preference labels consistent? Check inter-annotator agreement.
2. **Beta too high**: The model can't deviate from reference enough to learn preferences. Try lowering beta.
3. **Learning rate**: Too low = slow convergence. Too high = unstable.
4. **Sequence length normalization**: Are chosen and rejected responses similar in length? Length bias is common.
5. **Reference model quality**: If the SFT model is bad, the reference constrains you to bad behavior.
6. **Data diversity**: If all examples are similar, the model can't generalize.

In [ ]:
# Summary comparison: Radar chart
import matplotlib.patches as mpatches

categories = ['Simplicity', 'Sample\nEfficiency', 'Stability', 
              'Scalability', 'Data\nFlexibility']
n_cats = len(categories)

# Scores (1-5) for each method
methods = {
    'PPO':  [2, 4, 2, 5, 3],
    'DPO':  [5, 3, 4, 4, 3],
    'KTO':  [4, 3, 4, 4, 5],
    'GRPO': [4, 4, 4, 5, 3],
}

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

colors_radar = {'PPO': 'red', 'DPO': 'blue', 'KTO': 'green', 'GRPO': 'purple'}

for method, scores in methods.items():
    values = scores + scores[:1]  # Close the polygon
    ax.plot(angles, values, 'o-', linewidth=2, color=colors_radar[method], label=method)
    ax.fill(angles, values, alpha=0.1, color=colors_radar[method])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 5.5)
ax.set_title('Preference Optimization Methods Comparison', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.show()

print("Summary:")
print("  PPO  - Most powerful at scale, hardest to tune")
print("  DPO  - Best balance of simplicity and performance")
print("  KTO  - Best for unpaired data (production user feedback)")
print("  GRPO - Best for reasoning tasks (DeepSeek-R1 approach)")

---

## 8. Key Takeaways

### Foundation (Know These Cold)
- [ ] `compute_log_probs()`: Forward pass, shift logits, gather, mask, sum
- [ ] DPO: -log sigmoid(beta * (log_ratio_chosen - log_ratio_rejected))
- [ ] GRPO: Sample K, group-normalize rewards, PPO-clip update
- [ ] All methods need a reference model except ORPO

### Implementation (Be Able to Code)
- [ ] Full DPO training loop (4 forward passes per step)
- [ ] IPO loss (squared instead of sigmoid)
- [ ] KTO loss (unpaired, asymmetric good/bad)
- [ ] GRPO loss + group advantage computation

### Decision Framework (Interview Differentiator)
- [ ] Paired preference data available -> DPO or IPO
- [ ] Only thumbs up/down data -> KTO
- [ ] Need online learning / best performance at scale -> PPO
- [ ] Reasoning tasks / math / code -> GRPO
- [ ] Want to combine SFT + alignment -> ORPO
- [ ] Noisy preference labels -> IPO (robust to noise)

### Common Mistakes (Avoid These)
- [ ] Forgetting to freeze the reference model
- [ ] Not shifting logits by 1 when computing log-probs
- [ ] Not masking prompt tokens (only response should contribute to loss)
- [ ] Wrong beta (too high = no learning, too low = unstable)

---

**Next**: Tutorial 42 - Coding Agents from Scratch